<a href="https://colab.research.google.com/github/suryaph971/Langchain/blob/main/QA_on_private_documents(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install -r requirements\ \(1\).txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=6fefbc9e275465050fc17da49605ac9de16fb5c836a9ad04a9a0487bf9d0aea4
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langcha

In [3]:
import os
from dotenv import find_dotenv,load_dotenv
load_dotenv(find_dotenv(),override=True)


True

In [21]:
pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 11.6 MB/s  0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pinecone]


#Loading Documents

In [4]:
def load_document(file):
    import os
    name,extension = os.path.splitext(file)
    if extension == '.pdf':
        from langchain.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document type is not supported')
        return None

    data = loader.load()
    return data


In [5]:
#wikipedia
def load_from_wikipedia(query,lang='en',load_max_docs=2):
    from langchain.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query,lang=lang,load_max_docs=load_max_docs)
    data = loader.load()
    return data

#chunking data

In [6]:
def chunk_data(data,chunk_size=256):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

#Calculating Cost

In [17]:
def print_embedding_costs(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'Total Tokens : {total_tokens}')
    print(f'Embedding Cost in USD : {total_tokens/1000*0.00002:.6f}')

### Embedding and Uploading to a Vector Database (Pinecone)

In [9]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [10]:
pip install -q langchain_google_genai


In [11]:
def insert_or_fetch_embeddings(index_name,chunks):
    import pinecone
    from langchain_community.vectorstores import Pinecone
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()

    embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001',dimensions=1536)

    if index_name in pc.list_indexes().names():
        print(f'Index {index_name} already exists.Loading embeddings .....')
        vector_store = Pinecone.from_existing.index(index_name,embeddings)
        print('OK')
    else:
        print(f'Creating index {index_name} and embeddings ....')
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec = ServerlessSpec(
                cloud='aws',
                region='us-east-1'
            )
        )

        vector_store = Pinecone.from_documents(chunks,embeddings,index_name=index_name)
        print('OK')
        return vector_store





In [29]:
pip install -q chromadb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [41]:
def index_or_fetch_embeddings_chroma(chunks,persist_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model='text-embedding-004',dimensions=1536)
    vector_store = Chroma.from_documents(chunks,embeddings,persist_directory=persist_directory)
    return vector_store

In [42]:
def load_embeddings_chroma(persist_directory='./chroma_db'):
    from langchain.vectorstores import Chroma
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model='text-embedding-004',dimensions=1536)
    vector_store = Chroma(persist_directory=persist_directory,embedding_function=embeddings)
    return vector_store


In [12]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    if index_name=='all':
        indexes = pc.list_indexes()
        print("Deleting all indexes")
        for index in indexes:
            pc.delete_index(index)
            print('OK')
    else:
        print(f'Deleting index {index_name}')
        pc.delete_index(index_name)
        print('OK')


#Asking and Getting Answers

In [13]:
def ask_and_get_answer(vector_store,query,k=3):
    from langchain.chains import RetrievalQA
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash')
    retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={'k':k})

    chain = RetrievalQA.from_chain_type(llm=llm,chain_type='stuff',retriever=retriever)
    response = chain.invoke(query)
    return response


#Ask a PDF

In [14]:
data = load_document('us_constitution.pdf')

print(f'You have {len(data)} pages in your document')
print(f'There are {len(data[0].page_content)} characters in this page')

Loading us_constitution.pdf
You have 41 pages in your document
There are 639 characters in this page


In [15]:
chunks = chunk_data(data)
print(len(chunks))

224


In [18]:
print_embedding_costs(chunks)

Total Tokens : 9842
Embedding Cost in USD : 0.000197


In [44]:
vector_store = index_or_fetch_embeddings_chroma(chunks)

In [45]:
q="What is bill of rights?"
answer = ask_and_get_answer(vector_store,q)
print(answer)

{'query': 'What is bill of rights?', 'result': "Based on the provided text, the Bill of Rights includes the right to assemble and petition the government, and the right to keep and bear arms (Second Amendment).  It also includes protection against unreasonable searches and seizures.  The provided text only gives snippets, so it doesn't fully describe the Bill of Rights."}


In [46]:
import time
i=1
print('Write Quit or Exit to quit')
while True:
    q=input(f'Question #{i}')
    i=i+1
    if q.lower() in ['quit','exit']:
        print("Exiting")
        time.sleep(2)
        break
    else:
        answer = ask_and_get_answer(vector_store,q)
        print(f'\nAnswer: {answer}')
        print(f'\n {"-" * 50} \n')

Write Quit or Exit to quit
Question #1What is the first amendment described in the document?

Answer: {'query': 'What is the first amendment described in the document?', 'result': 'Congress shall make no law respecting an establishment of religion, or prohibiting the free exercise thereof; or abridging the freedom of speech, or of the press; or the right of the people peaceably to assemble, and to petition the Government for a redress of grievances.'}

 -------------------------------------------------- 

Question #2what about the second amendment?

Answer: {'query': 'what about the second amendment?', 'result': 'The Second Amendment states:  "A well regulated Militia, being necessary to the security of a free State, the right of the people to keep and bear Arms, shall not be infringed."'}

 -------------------------------------------------- 

Question #3quit
Exiting


In [23]:
!pip uninstall -y pinecone-client

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0


In [24]:
!pip install pinecone

In [25]:
def insert_or_fetch_embeddings(index_name,chunks):
    import pinecone
    from langchain_community.vectorstores import Pinecone
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()

    embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001',dimensions=1536)

    if index_name in pc.list_indexes().names():
        print(f'Index {index_name} already exists.Loading embeddings .....')
        vector_store = Pinecone.from_existing_index(index_name,embeddings)
        print('OK')
    else:
        print(f'Creating index {index_name} and embeddings ....')
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec = ServerlessSpec(
                cloud='aws',
                region='us-east-1'
            )
        )

        vector_store = Pinecone.from_documents(chunks,embeddings,index_name=index_name)
        print('OK')
        return vector_store

In [26]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    if index_name=='all':
        indexes = pc.list_indexes()
        print("Deleting all indexes")
        for index in indexes:
            pc.delete_index(index)
            print('OK')
    else:
        print(f'Deleting index {index_name}')
        pc.delete_index(index_name)
        print('OK')